In [15]:
import os
import numpy as np
# uncomment to disable NVIDIA GPUs
#os.environ['CUDA_VISIBLE_DEVICES'] = ''
# or pick the device (cpu, gpu, and tpu)
#os.environ['JAX_PLATFORMS'] = 'cpu'

# change JAX GPU memory preallocation fraction
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '.95'

# you do not want this
#os.environ['XLA_FLAGS'] = '--xla_gpu_deterministic_ops=true'

import jax

import jax.numpy as jnp
from jax import jit, lax
#jax.print_environment_info()

!nvidia-smi --query-gpu=gpu_name --format=csv,noheader

import matplotlib.pyplot as plt
import matplotlib_inline


matplotlib_inline.backend_inline.set_matplotlib_formats('jpeg')

from pmwd import (
    Configuration,
    Cosmology, SimpleLCDM,
    boltzmann, linear_power, growth,
    white_noise, linear_modes,
    lpt,
    nbody,
    scatter,
)
from pmwd.nbody import nbody_step, nbody_init
from pmwd.pm_util import fftinv
from pmwd.spec_util import powspec
from pmwd.vis_util import simshow

/bin/bash: linha 1: nvidia-smi: comando não encontrado


In [10]:
if jax.default_backend() == 'gpu':
    ptcl_spacing = 1.  # Lagrangian space Cartesian particle grid spacing, in Mpc/h by default
    ptcl_grid_shape = (128,) * 3
else:
    ptcl_spacing = 4.
    #ptcl_grid_shape = (64,) * 3
    ptcl_grid_shape = (128,) * 3

conf = Configuration(ptcl_spacing, ptcl_grid_shape, mesh_shape=2)  # 2x mesh shape

print(conf)  # with other default parameters
print(f'\n Simulating {conf.ptcl_num} particles with a {conf.mesh_shape} mesh for {conf.a_nbody_num} time steps.')

Configuration(ptcl_spacing=4.0,
              ptcl_grid_shape=(128, 128, 128),
              mesh_shape=(256, 256, 256),
              cosmo_dtype=dtype('float64'),
              pmid_dtype=dtype('int16'),
              float_dtype=dtype('float32'),
              k_pivot_Mpc=0.05,
              T_cmb=2.7255,
              M=1.98847e+40,
              L=3.0856775815e+22,
              T=3.0856775815e+17,
              transfer_fit=True,
              transfer_fit_nowiggle=False,
              transfer_lgk_min=-4,
              transfer_lgk_max=3,
              transfer_lgk_maxstep=0.0078125,
              growth_rtol=1.4901161193847656e-08,
              growth_atol=1.4901161193847656e-08,
              growth_inistep=(1, None),
              lpt_order=2,
              a_start=0.015625,
              a_stop=1,
              a_lpt_maxstep=0.0078125,
              a_nbody_maxstep=0.015625,
              symp_splits=((0, 0.5), (1, 0.5)),
              chunk_size=16777216)

 Simulating 2097

In [ ]:
def extract_phase_space_simple(modes, cosmo, conf, a_values=None, output_dir='phase_space_data'):
    """
    Extrai espaços de fase para diferentes valores de a e salva em arquivos de texto.
    
    Parameters:
    -----------
    modes : array
        Modos iniciais
    cosmo : Cosmology
        Parâmetros cosmológicos
    conf : Configuration
        Configuração da simulação
    a_values : list, optional
        Lista de valores de a para extrair espaços de fase. Se None, usa valores padrão.
    output_dir : str
        Diretório para salvar os arquivos
    """
    
    # Pra nao precisar passar valores de a 
    if a_values is None:
        a_values = [1/64, 1/32, 1/16, 1/8, 1/4, 1/2, 1.0]
    
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    print(f"Extraindo espaços de fase para {len(a_values)} valores de a...")
    
    cosmo = jax.block_until_ready(boltzmann(cosmo, conf))
    modes = linear_modes(modes, cosmo, conf)
    ptcl, obsvbl = jax.block_until_ready(lpt(modes, cosmo, conf))
    
    ptcl, obsvbl = nbody_init(conf.a_nbody[0], ptcl, obsvbl, cosmo, conf)
    
    a_indices = [jnp.argmin(jnp.abs(conf.a_nbody - a_target)) for a_target in a_values]
    
    for i, a_idx in enumerate(a_indices):
        actual_a = conf.a_nbody[a_idx]
        
        print(f"Processando a = {actual_a:.6f} ({i+1}/{len(a_values)})")
        
        if a_idx == 0:
            ptcl_current = ptcl
        else:
            ptcl_temp = ptcl
            obsvbl_temp = obsvbl
            
            for step in range(a_idx):
                a_prev = conf.a_nbody[step]
                a_next = conf.a_nbody[step + 1]
                ptcl_temp, obsvbl_temp = jax.block_until_ready(
                    nbody_step(a_prev, a_next, ptcl_temp, obsvbl_temp, cosmo, conf)
                )
            
            ptcl_current = ptcl_temp
        
        dens_3d = scatter(ptcl_current, conf)
        
        phase_space = jnp.zeros(tuple(2*s for s in conf.mesh_shape), dtype=conf.float_dtype)
        phase_space = scatter(ptcl_current, conf, mesh=phase_space, val=1, cell_size=conf.cell_size / 2)
        phase_space_2d = phase_space.sum(axis=2)  # projeta no eixo z
        
        dens_filename = f'density_3d_a_{actual_a:.6f}.txt'
        dens_filepath = os.path.join(output_dir, dens_filename)
        np.savetxt(dens_filepath, dens_3d.reshape(-1), fmt='%.6e')
        
        phase_filename = f'phase_space_a_{actual_a:.6f}.txt'
        phase_filepath = os.path.join(output_dir, phase_filename)
        np.savetxt(phase_filepath, phase_space_2d, fmt='%.6e')
        
        info_filename = f'info_a_{actual_a:.6f}.txt'
        info_filepath = os.path.join(output_dir, info_filename)
        with open(info_filepath, 'w') as f:
            f.write(f"# output data information\n")
            f.write(f"a = {actual_a:.6f}\n")
            f.write(f"density_3d_shape = {dens_3d.shape}\n")
            f.write(f"phase_space_2d_shape = {phase_space_2d.shape}\n")
            f.write(f"mesh_shape = {conf.mesh_shape}\n")
            f.write(f"cell_size = {conf.cell_size}\n")
            f.write(f"ptcl_num = {conf.ptcl_num}\n")
        
        print(f"  Salvou dados em {output_dir}/")
        print(f"    - {dens_filename}")
        print(f"    - {phase_filename}")
        print(f"    - {info_filename}")
    
    print(f"\\n Processamento concluído. {len(a_indices)} salvos em {output_dir} /")


In [ ]:

cosmo = Cosmology(conf, A_s_1e9=2.1, n_s=0.96, Omega_m=0.31, Omega_b=0.05, h=0.67, xi_=0.0, w_0_=-1.0, w_a_= 0.0)
modes = white_noise(1, conf)
a_values = [1.0, 1/2]

extract_phase_space_simple(modes, cosmo, conf, a_values=a_values, output_dir='phase_space_data')


Extraindo espaços de fase para 2 valores de a...
Processando a = 1.000000 (1/2)
Processando a = 1.000000 (1/2)
  Salvou dados em phase_space_data/
    - density_3d_a_1.000000.txt
    - phase_space_a_1.000000.txt
    - info_a_1.000000.txt
Processando a = 0.500000 (2/2)
  Salvou dados em phase_space_data/
    - density_3d_a_1.000000.txt
    - phase_space_a_1.000000.txt
    - info_a_1.000000.txt
Processando a = 0.500000 (2/2)
  Salvou dados em phase_space_data/
    - density_3d_a_0.500000.txt
    - phase_space_a_0.500000.txt
    - info_a_0.500000.txt
\nProcessamento concluído. 2 snapshots salvos em phase_space_data/
  Salvou dados em phase_space_data/
    - density_3d_a_0.500000.txt
    - phase_space_a_0.500000.txt
    - info_a_0.500000.txt
\nProcessamento concluído. 2 snapshots salvos em phase_space_data/


In [ ]:
# listagem do chatgpt, n ficou bom. 
# codigo prolixo 

def list_phase_space_files(data_dir='phase_space_data'):
    """
    Lista arquivos de espaço de fase salvos.
    
    Parameters:
    -----------
    data_dir : str
        Diretório onde estão os arquivos de dados
    """
    import os
    
    if not os.path.exists(data_dir):
        print(f"Diretório {data_dir} não encontrado.")
        return
    
    files = os.listdir(data_dir)
    files.sort()
    
    # Agrupa arquivos por valor de a
    a_values = set()
    for file in files:
        if '_a_' in file:
            a_part = file.split('_a_')[1]
            a_value = a_part.split('.txt')[0]
            a_values.add(float(a_value))
    
    print(f"Arquivos encontrados em {data_dir}:")
    for a_val in sorted(a_values):
        print(f"\\n  a = {a_val:.6f}:")
        for file in files:
            if f'_a_{a_val:.6f}.txt' in file:
                print(f"    - {file}")

def load_phase_space_snapshot(data_dir, a_value):
    """
    Carrega um snapshot específico de espaço de fase.
    
    Parameters:
    -----------
    data_dir : str
        Diretório onde estão os arquivos
    a_value : float
        Valor de a para carregar
    
    Returns:
    --------
    data : dict
        Dados carregados
    """
    import numpy as np
    import os
    
    dens_file = os.path.join(data_dir, f'density_3d_a_{a_value:.6f}.txt')
    phase_file = os.path.join(data_dir, f'phase_space_a_{a_value:.6f}.txt')
    info_file = os.path.join(data_dir, f'info_a_{a_value:.6f}.txt')
    
    if not all(os.path.exists(f) for f in [dens_file, phase_file, info_file]):
        print(f"Nem todos os arquivos encontrados para a = {a_value:.6f}")
        return None
    
    info = {}
    with open(info_file, 'r') as f:
        content = f.read().strip()
        lines = content.split('\\n')
        
        for line in lines:
            if '=' in line and not line.startswith('#'):
                key, value = line.strip().split(' = ', 1)  # Use maxsplit=1 to handle values with =
                if key == 'a':
                    info[key] = float(value)
                elif 'shape' in key:
                    # Remove parênteses e converte para tupla
                    shape_str = value.strip('()')
                    info[key] = tuple(map(int, shape_str.split(', ')))
                else:
                    try:
                        info[key] = float(value)
                    except:
                        info[key] = value
    
    print(f"Debug: info keys = {list(info.keys())}")  # Debug
    
    density_flat = np.loadtxt(dens_file)
    density_3d = density_flat.reshape(info['density_3d_shape'])
    
    phase_space_2d = np.loadtxt(phase_file)
    
    return {
        'a': info['a'],
        'density_3d': density_3d,
        'phase_space_2d': phase_space_2d,
        'info': info
    }

list_phase_space_files('phase_space_data')

print("\\n" + "="*50)
print("Exemplo de carregamento de snapshot:")
data = load_phase_space_snapshot('phase_space_data', 1.0)
if data:
    print(f"Carregado snapshot para a = {data['a']}")
    print(f"Densidade 3D shape: {data['density_3d'].shape}")
    print(f"Espaço de fase shape: {data['phase_space_2d'].shape}")
    print(f"Valor mínimo densidade: {data['density_3d'].min():.6e}")
    print(f"Valor máximo densidade: {data['density_3d'].max():.6e}")
    print(f"Valor mínimo phase space: {data['phase_space_2d'].min():.6e}")
    print(f"Valor máximo phase space: {data['phase_space_2d'].max():.6e}")

Arquivos encontrados em phase_space_data:
\n  a = 0.500000:
    - density_3d_a_0.500000.txt
    - info_a_0.500000.txt
    - phase_space_a_0.500000.txt
\n  a = 1.000000:
    - density_3d_a_1.000000.txt
    - info_a_1.000000.txt
    - phase_space_a_1.000000.txt
\n==================================================
Exemplo de carregamento de snapshot:
Debug: info keys = ['a', 'density_3d_shape', 'phase_space_2d_shape', 'mesh_shape', 'cell_size', 'ptcl_num']
Carregado snapshot para a = 1.0
Densidade 3D shape: (256, 256, 256)
Espaço de fase shape: (512, 512)
Valor mínimo densidade: 0.000000e+00
Valor máximo densidade: 8.789062e+02
Valor mínimo phase space: 1.347589e-01
Valor máximo phase space: 1.045050e+02
Carregado snapshot para a = 1.0
Densidade 3D shape: (256, 256, 256)
Espaço de fase shape: (512, 512)
Valor mínimo densidade: 0.000000e+00
Valor máximo densidade: 8.789062e+02
Valor mínimo phase space: 1.347589e-01
Valor máximo phase space: 1.045050e+02


In [ ]:
print("\\n" + "="*50)
print("Teste de carregamento de ambos os snapshots:")

for a_val in [1.0, 0.5]:
    print(f"\\nCarregando a = {a_val}:")
    data = load_phase_space_snapshot('phase_space_data', a_val)
    if data:
        print(f"  ✓ Sucesso! a = {data['a']}")
        print(f"  ✓ Densidade 3D shape: {data['density_3d'].shape}")
        print(f"  ✓ Espaço de fase shape: {data['phase_space_2d'].shape}")
    else:
        print(f"  ✗ Falhou ao carregar")

\n==================================================
Teste de carregamento de ambos os snapshots:
\nCarregando a = 1.0:
Debug: info keys = ['a', 'density_3d_shape', 'phase_space_2d_shape', 'mesh_shape', 'cell_size', 'ptcl_num']
  ✓ Sucesso! a = 1.0
  ✓ Densidade 3D shape: (256, 256, 256)
  ✓ Espaço de fase shape: (512, 512)
\nCarregando a = 0.5:
Debug: info keys = ['a', 'density_3d_shape', 'phase_space_2d_shape', 'mesh_shape', 'cell_size', 'ptcl_num']
  ✓ Sucesso! a = 0.5
  ✓ Densidade 3D shape: (256, 256, 256)
  ✓ Espaço de fase shape: (512, 512)
